# Integrated Project — NumPy + Pandas + Matplotlib + Seaborn

Run the cells from top to bottom.

## Project: Employee Analytics & Data Visualization

This project combines **NumPy + Pandas + Matplotlib + Seaborn** in one end-to-end workflow.

### Business questions

1. What is the salary distribution?
2. Which departments have the highest average salaries?
3. Is experience related to salary?
4. Are there missing values, duplicates or outliers?
5. How do performance and training relate to salary?
6. Which features are strongly correlated?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("data/employee_analytics_project.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

default_color = plt.rcParams["axes.prop_cycle"].by_key()["color"][0]

## 1. Initial Data Inspection

In [ ]:
print("Columns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df.info()

## 2. Statistical Summary

In [ ]:
df.describe(include="all")

## 3. NumPy Analysis on Numerical Columns

In [ ]:
salary_np = df["Monthly_Salary"].dropna().to_numpy()

print("Count:", salary_np.size)
print("Mean:", np.mean(salary_np))
print("Median:", np.median(salary_np))
print("Standard deviation:", np.std(salary_np))
print("Min:", np.min(salary_np))
print("Max:", np.max(salary_np))

q1, q3 = np.percentile(salary_np, [25, 75])
iqr = q3 - q1

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)

## 4. Remove Duplicates

In [ ]:
df_clean = df.drop_duplicates(subset=["Employee_ID"]).copy()

print("Before:", len(df))
print("After:", len(df_clean))

## 5. Handle Missing Values

In [ ]:
# Numerical strategy:
# - Age: median
# - Performance score: department median
# - Training hours: median
# - Salary: department median

df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())

perf_median = df_clean.groupby("Department")["Performance_Score"].transform("median")
df_clean["Performance_Score"] = df_clean["Performance_Score"].fillna(perf_median)
df_clean["Performance_Score"] = df_clean["Performance_Score"].fillna(df_clean["Performance_Score"].median())

df_clean["Training_Hours"] = df_clean["Training_Hours"].fillna(df_clean["Training_Hours"].median())

salary_median = df_clean.groupby("Department")["Monthly_Salary"].transform("median")
df_clean["Monthly_Salary"] = df_clean["Monthly_Salary"].fillna(salary_median)
df_clean["Monthly_Salary"] = df_clean["Monthly_Salary"].fillna(df_clean["Monthly_Salary"].median())

df_clean["Annual_Salary"] = df_clean["Monthly_Salary"] * 12

print(df_clean.isnull().sum())

## 6. Outlier Detection with IQR

In [ ]:
q1 = df_clean["Monthly_Salary"].quantile(0.25)
q3 = df_clean["Monthly_Salary"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

salary_outliers = df_clean[
    (df_clean["Monthly_Salary"] < lower_bound) |
    (df_clean["Monthly_Salary"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Outlier count:", len(salary_outliers))
salary_outliers[["Employee_ID", "Department", "Experience_Years", "Monthly_Salary"]]

## 7. Create Derived Features with NumPy

In [ ]:
conditions = [
    df_clean["Experience_Years"] < 3,
    df_clean["Experience_Years"].between(3, 7),
    df_clean["Experience_Years"].between(8, 14),
    df_clean["Experience_Years"] >= 15
]
choices = ["Entry", "Mid", "Senior", "Expert"]

df_clean["Experience_Level"] = np.select(conditions, choices, default="Unknown")

df_clean["High_Performer"] = np.where(
    df_clean["Performance_Score"] >= 4.2,
    "Yes",
    "No"
)

df_clean[["Experience_Years", "Experience_Level", "Performance_Score", "High_Performer"]].head(10)

## 8. Department Summary with Pandas GroupBy

In [ ]:
department_summary = (
    df_clean.groupby("Department", as_index=False)
    .agg(
        Employee_Count=("Employee_ID", "count"),
        Avg_Age=("Age", "mean"),
        Avg_Experience=("Experience_Years", "mean"),
        Avg_Performance=("Performance_Score", "mean"),
        Avg_Monthly_Salary=("Monthly_Salary", "mean"),
        Total_Projects=("Projects_Completed", "sum")
    )
)

department_summary = department_summary.round(2)
department_summary

## 9. City Summary

In [ ]:
city_summary = (
    df_clean.groupby("City", as_index=False)
    .agg(
        Employees=("Employee_ID", "count"),
        Avg_Salary=("Monthly_Salary", "mean"),
        Avg_Performance=("Performance_Score", "mean")
    )
    .round(2)
)

city_summary

## 10. Matplotlib — Salary Distribution

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df_clean["Monthly_Salary"], bins=15)
plt.xlabel("Monthly Salary")
plt.ylabel("Employee Count")
plt.title("Monthly Salary Distribution")
plt.show()

## 11. Matplotlib — Average Salary by Department

In [ ]:
dept_plot = department_summary.sort_values("Avg_Monthly_Salary", ascending=False)

plt.figure(figsize=(9, 5))
plt.bar(dept_plot["Department"], dept_plot["Avg_Monthly_Salary"])
plt.xlabel("Department")
plt.ylabel("Average Monthly Salary")
plt.title("Average Salary by Department")
plt.show()

## 12. Matplotlib — Experience vs Salary

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df_clean["Experience_Years"], df_clean["Monthly_Salary"])
plt.xlabel("Experience (Years)")
plt.ylabel("Monthly Salary")
plt.title("Experience vs Monthly Salary")
plt.show()

## 13. Seaborn — Count Plot

In [ ]:
sns.countplot(data=df_clean, x="Department", hue="Department", legend=False)
plt.title("Employee Count by Department")
plt.show()

## 14. Seaborn — Histogram + KDE

In [ ]:
sns.histplot(data=df_clean, x="Monthly_Salary", kde=True, color=default_color)
plt.title("Salary Distribution with KDE")
plt.show()

## 15. Seaborn — Box Plot for Outliers

In [ ]:
sns.boxplot(data=df_clean, x="Department", y="Monthly_Salary", hue="Department", legend=False)
plt.title("Salary Distribution by Department")
plt.show()

## 16. Seaborn — Experience vs Salary with Department

In [ ]:
sns.scatterplot(
    data=df_clean,
    x="Experience_Years",
    y="Monthly_Salary",
    hue="Department"
)
plt.title("Experience vs Salary by Department")
plt.show()

## 17. Seaborn — Performance vs Salary

In [ ]:
sns.regplot(
    data=df_clean,
    x="Performance_Score",
    y="Monthly_Salary"
)
plt.title("Performance Score vs Salary")
plt.show()

## 18. Correlation Matrix

In [ ]:
numeric_cols = [
    "Age",
    "Experience_Years",
    "Performance_Score",
    "Projects_Completed",
    "Training_Hours",
    "Monthly_Salary"
]

correlation = df_clean[numeric_cols].corr()
correlation

## 19. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True)
plt.title("Feature Correlation Heatmap")
plt.show()

## 20. Pair Plot

In [ ]:
sns.pairplot(
    df_clean[
        [
            "Experience_Years",
            "Performance_Score",
            "Projects_Completed",
            "Training_Hours",
            "Monthly_Salary"
        ]
    ],
    plot_kws={"color": default_color},
    diag_kws={"color": default_color}
)
plt.show()

## 21. Department + Experience Level Pivot Table

In [ ]:
pivot = pd.pivot_table(
    df_clean,
    values="Monthly_Salary",
    index="Department",
    columns="Experience_Level",
    aggfunc="mean",
    fill_value=0
).round(2)

pivot

## 22. Top 10 Highest Salaries

In [ ]:
top_10 = (
    df_clean[
        ["Employee_ID", "Department", "City", "Experience_Years", "Performance_Score", "Monthly_Salary"]
    ]
    .sort_values("Monthly_Salary", ascending=False)
    .head(10)
)

top_10

## 23. Save Cleaned Data and Summary

In [ ]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

df_clean.to_csv(output_dir / "cleaned_employee_data.csv", index=False)
department_summary.to_csv(output_dir / "department_summary.csv", index=False)
city_summary.to_csv(output_dir / "city_summary.csv", index=False)

print("Saved:")
print(output_dir / "cleaned_employee_data.csv")
print(output_dir / "department_summary.csv")
print(output_dir / "city_summary.csv")

## 24. Final Project Summary

This notebook used all four libraries together:

### NumPy
- Arrays
- Mean / median / standard deviation
- Percentiles / IQR
- `np.select`
- `np.where`

### Pandas
- CSV loading
- Inspection
- Missing-value handling
- Duplicate removal
- GroupBy
- Aggregation
- Pivot table
- Sorting
- Feature creation
- CSV export

### Matplotlib
- Histogram
- Bar chart
- Scatter plot

### Seaborn
- Count plot
- Histogram + KDE
- Box plot
- Scatter plot with hue
- Regression plot
- Heatmap
- Pair plot

This is a complete pre-Machine-Learning EDA workflow.